#**Definindo catalogo Usuário X Filmes**

* **df:** matriz 50 × 20 completa (ratings inteiros de 1 a 5)

* **df_com_nan:** mesma matriz com ~20% de valores faltantes

* **usuarios:** (50 nomes) e filmes (20 títulos) como listas auxiliares

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# -------- 1. DADOS --------
np.random.seed(42)

# 20 filmes organizados por "perfil" (para gerar padrões coerentes)
filmes = [
    # Ficção científica / Ação
    'Star Wars', 'Matrix', 'Interestelar', 'Blade Runner 2049', 'Duna',
    # Romance / Drama
    'Titanic', 'Diário de uma Paixão', 'Simplesmente Amor', 'Orgulho e Preconceito', 'La La Land',
    # Clássicos / Cult
    'O Poderoso Chefão', 'Pulp Fiction', 'Cidade de Deus', 'Clube da Luta', 'Forrest Gump',
    # Animação / Família
    'Toy Story', 'Procurando Nemo', 'Shrek', 'Up: Altas Aventuras', 'Soul',
    # Terror / Suspense
    'O Iluminado', 'Psicose', 'Corra!', 'Hereditário', 'Seven'
]

# 50 usuários
usuarios = [
    'Ana', 'Bruno', 'Carla', 'Diego', 'Elisa',
    'Felipe', 'Gabriela', 'Henrique', 'Isabela', 'João',
    'Karina', 'Lucas', 'Mariana', 'Nathan', 'Olívia',
    'Paulo', 'Queren', 'Rafael', 'Sofia', 'Thiago',
    'Ursula', 'Victor', 'Wesley', 'Xênia', 'Yuri',
    'Zélia', 'Alice', 'Beto', 'Cecília', 'Davi',
    'Eduarda', 'Fábio', 'Giovana', 'Heitor', 'Iara',
    'Júlia', 'Kevin', 'Larissa', 'Miguel', 'Natália',
    'Otávio', 'Priscila', 'Renato', 'Sabrina', 'Tomás',
    'Ubiratã', 'Valéria', 'Wagner', 'Yasmin', 'Zeca'
]


# -------- GERAR MATRIZ DE RATINGS COM PADRÕES --------
n_filmes = len(filmes)
n_usuarios = len(usuarios)

# Perfis de preferência (gostos latentes) por grupo de filmes
# Blocos: [SciFi/Ação, Romance, Clássicos, Animação, Terror]
perfil_por_filme = np.array([0]*5 + [1]*5 + [2]*5 + [3]*5 + [4]*5)

# Cada usuário tem preferências por gênero (vetor de pesos)
preferencias_usuarios = np.random.dirichlet(
    alpha=[1.2, 1.2, 1.2, 1.2, 1.2], size=n_usuarios
)  # shape (50, 5)

# Rating base = soma ponderada das preferências + ruído
ratings = np.zeros((n_usuarios, n_filmes))
for u in range(n_usuarios):
    for f in range(n_filmes):
        base = 5.0 * preferencias_usuarios[u, perfil_por_filme[f]] * 5  # escala 0-5
        ruido = np.random.normal(0, 0.7)
        ratings[u, f] = np.clip(round(base + ruido), 1, 5)

# -------- DATAFRAME COMPLETO (SEM NAN) --------
df = pd.DataFrame(ratings.astype(int), index=usuarios, columns=filmes)

# -------- CRIAR VERSÃO COM NaN (~20% ausente) --------
df_com_nan = df.copy()
n_remover = int(0.20 * df.size)  # ~200 valores
posicoes = np.random.choice(df.size, size=n_remover, replace=False)
for pos in posicoes:
    u = pos // n_filmes
    f = pos % n_filmes
    df_com_nan.iloc[u, f] = np.nan



In [8]:
print("\nDataFrame com valores ausentes (simulando esparsidade):")
print(df_com_nan.iloc[:3,:10])


DataFrame com valores ausentes (simulando esparsidade):
       Star Wars  Matrix  Interestelar  Blade Runner 2049  Duna  Titanic  \
Ana          5.0     NaN           5.0                4.0   5.0      2.0   
Bruno        5.0     5.0           5.0                NaN   5.0      2.0   
Carla        NaN     1.0           2.0                1.0   1.0      NaN   

       Diário de uma Paixão  Simplesmente Amor  Orgulho e Preconceito  \
Ana                     3.0                3.0                    NaN   
Bruno                   NaN                1.0                    1.0   
Carla                   5.0                5.0                    5.0   

       La La Land  
Ana           2.0  
Bruno         3.0  
Carla         NaN  


**Atividade**

Construa um sistema de recomendação baseado no algorítimo SVD, a fim de recomendar 3 filmes para os usuários ['Nathan', 'Alice', 'Yuri', 'Larissa']